# Convergence to $\pi$ in Wasserstein and total variation
## NALD with $J=0$, constant $J_a$, and state-dependent $J_s(x)$

We measure how fast the **law of $X_t$** approaches the target, rather than how fast a single
trajectory decorrelates: launch $M$ independent chains from a common point mass $\delta_{x_0}$ and, at
a sequence of times, estimate the distance between the ensemble $\mathrm{Law}(X_t)$ and
$\pi\propto e^{-U}$.

$$\widehat{\mu}_t=\frac1M\sum_{k=1}^M\delta_{X_t^{(k)}},\qquad
  \text{plot } \ \mathcal{W}_1(\widehat\mu_t,\pi)\ \text{ and }\ \mathrm{TV}(\widehat\mu_t,\pi)\ \text{ vs } t .$$

$\pi$ is represented by $M$ **exact i.i.d. draws** (both targets are exactly sampleable), so no density
estimate or reference chain is involved.

### Estimators

Neither distance is directly computable between empirical measures in $\mathbb{R}^3$ at this sample
size, so both are **sliced** — projected onto random directions, where each is exact in one dimension.

* **Sliced $\mathcal{W}_1$.** In one dimension, $\mathcal{W}_1$ between two equal-size empirical
  samples is exactly the mean absolute difference of their order statistics. We average over $L=64$
  fixed directions, drawn once and shared by every method and every time, in standardised coordinates
  $y=x/\mathrm{sd}_\pi(x)$ so all three modes count equally. Sliced $\mathcal{W}_1$ is a genuine metric.
* **Total variation.** TV between continuous laws cannot be estimated from samples without
  discretising, so we report the **binned** TV on $B=50$ equiprobable reference bins. This is a *lower
  bound* on the true TV for every $B$, increasing to it as $B\to\infty$.

### Two floors, both plotted

1. **Finite-sample floor** — even if $\mathrm{Law}(X_t)=\pi$ exactly, $\widehat\mu_t$ has $M$ samples,
   so the estimator sits at $O(M^{-1/2})$. Measured directly by applying the identical estimator to two
   *independent* exact samples, and drawn as a shaded band.
2. **Discretisation floor** — the chains sample $\pi_h$, not $\pi$. Whatever the curves plateau at
   above the band is that $O(h)$ bias.

Only the descent between these floors is a statement about the dynamics.

In [ ]:
import sys, time, math
sys.path.insert(0, "..")               # nald.py lives at the repository root

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from nald import (EllipticLaplace, L1Laplace, hat, nald, state_scale,
                  surrogate_gap, optimal_axis)

%matplotlib inline
np.set_printoptions(precision=4, suppress=True, linewidth=140)
plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 120, "font.size": 9,
    "axes.grid": True, "grid.alpha": 0.25, "axes.spines.top": False,
    "axes.spines.right": False, "legend.frameon": False, "figure.facecolor": "white",
})
COL = {"J0": "#4C566A", "Jdem": "#B48EAD", "Ja": "#BF616A", "Js": "#5E81AC"}

tgtA = EllipticLaplace(np.diag([9.0, 1.0, 0.25]), 0.1)   # U  = sqrt(x' S^-1 x),  U0 = sqrt(.. + delta^2)
tgtB = L1Laplace([2.0, 1.0, 0.4], 0.1)                   # U  = sum |x_i|/b_i,    U0 = sum sqrt(x_i^2+delta^2)/b_i
TARGETS = {"A": tgtA, "B": tgtB}
TNAME   = {"A": "elliptical Laplace", "B": r"$\ell^1$ Laplace"}
S_STATE = {k: state_scale(t) for k, t in TARGETS.items()}
SD      = {k: np.sqrt(np.diag(t.cov_exact())) for k, t in TARGETS.items()}
U_DEM   = np.ones(3)/np.sqrt(3)                          # the ad-hoc "democratic" axis

for k in TARGETS:
    print(f"target {k} ({TNAME[k]:<20}) marginal sd = {np.round(SD[k],3)}   s = {S_STATE[k]:.4f}")

---
# 1. Designing the constant perturbation $J_a$

A constant skew $J$ in $d=3$ is a cross-product operator, $J_av=u\times v$, so the only design freedom
is the **axis** $u$. The choice is not innocuous, and picking it arbitrarily is what limits the gain.

Linearise around the target's second-order structure: with $S=\mathrm{Cov}_\pi^{-1}$, the perturbed
drift matrix of the Gaussian surrogate is $(I+\alpha J)S$, whose relaxation rate is
$\min\mathrm{Re}\,\mathrm{spec}\bigl((I+\alpha J)S\bigr)$. Because $J$ is skew and $S$ symmetric,
$\mathrm{tr}(JS)=0$, so the three eigenvalues **always sum to $\mathrm{tr}(S)$** no matter what $J$ or
$\alpha$ is. Hence for every admissible perturbation

$$\min\mathrm{Re}\,\mathrm{spec}\bigl((I+\alpha J)S\bigr)\;\le\;\frac{\mathrm{tr}(S)}{3},$$

a hard ceiling, attained exactly when the perturbation **equalises the three relaxation rates**. The
best achievable improvement over the reversible dynamics is therefore
$\frac{\mathrm{tr}(S)/3}{\lambda_{\min}(S)}$ — $15.3\times$ for target A and $10\times$ for target B.

The surrogate is only a heuristic here: both potentials grow *linearly*, so $\nabla U_0$ is homogeneous
of degree zero rather than linear, and the relaxation is not exponential in the OU sense. We therefore
use the surrogate to fix the **direction** $u$ only, and choose the **strength** $\alpha$ by
measurement in §2. The old "democratic" axis $u=(1,1,1)/\sqrt3$ is kept throughout as a reference so
the size of the design effect is visible.

In [ ]:
DESIGN = {}
print("Gaussian-surrogate spectral gap of (I + alpha*hat(u)) Cov^-1\n")
for k, tg in TARGETS.items():
    a_star, u_star, ceiling, base = optimal_axis(tg.cov_exact())
    DESIGN[k] = dict(alpha_star=a_star, u=u_star, ceiling=ceiling, base=base)
    print(f"  target {k}:  Cov = {np.diag(tg.cov_exact())}")
    print(f"    reversible gap  lambda_min(S) = {base:.4f}")
    print(f"    ceiling         tr(S)/3       = {ceiling:.4f}   ({ceiling/base:.1f}x, unbeatable by ANY skew J)")
    print(f"    optimal axis    u*            = {np.round(u_star,4)}   first attains the ceiling at alpha = {a_star:.3f}")
    print(f"    {'alpha':>8}" + "".join(f"{a:>12g}" for a in [1,2,3,4,6,8,12]))
    S = np.linalg.inv(tg.cov_exact())
    for lab, u in [("democratic", U_DEM), ("optimal", u_star)]:
        g = [surrogate_gap(u, S, a) for a in [1,2,3,4,6,8,12]]
        print(f"    {lab:>8}" + "".join(f"{x/base:>11.2f}x" for x in g))
    print()

---
# 2. Choosing $\alpha$ by measurement

Two quantities decide the operating point, and both are measured rather than assumed.

* **Discretisation bias** — the *paired stationarity drift*: start $M$ chains at exact draws
  $X_0\sim\pi$, integrate to time $T$, and report $\max_f\bigl|\mathbb{E}f(X_T)-\mathbb{E}f(X_0)\bigr|
  /|\mathbb{E}f(X_0)|$. Zero for a scheme that preserves $\pi$; what is left is the $O(h)$ error, which
  grows with $\alpha$. Configurations above `TOL = 4%` (about three times the diagnostic's own noise
  floor) are rejected.
* **Worst-mode integrated autocorrelation time** $\max_f\tau_f$ over $f\in\{x_1,x_2,x_3,U\}$. Overall
  convergence is governed by the *slowest* mode, so this is the quantity to minimise — not the average,
  and not the mode a given perturbation happens to be good at.

**Selection rule, applied uniformly to both perturbations:** take the $\alpha$ minimising
$\max_f\tau_f$ among those whose measured bias is within `TOL`.

In [ ]:
def _acf(y):
    n = len(y); m = 1
    while m < 2*n: m *= 2
    f = np.fft.rfft(y - y.mean(), n=m)
    ac = np.fft.irfft(f*np.conj(f), n=m)[:n].real
    return ac/ac[0]

def iact(col, c=5.0):
    ac = np.mean([_acf(col[:, k]) for k in range(col.shape[1])], axis=0)
    t = 2*np.cumsum(ac) - 1.0
    for w in range(len(t)):
        if w >= c*t[w]: return max(float(t[w]), 1.0)
    return max(float(t[-1]), 1.0)

def stat_drift(tgt, T=2.0, h=0.005, M=80_000, seed=3, **kw):
    rng = np.random.default_rng(seed); X0 = tgt.sample(M, rng)
    _, XT = nald(tgt, h=h, n_steps=int(round(T/h)), n_chains=M, seed=seed+1,
                 x0=X0, store=False, block=25, **kw)
    F = lambda z: np.column_stack([z**2, tgt.U(z)[:, None]])
    F0, D = F(X0), F(XT) - F(X0)
    base = np.abs(F0.mean(0))
    return float(np.abs(D.mean(0)/base).max()), float((2*D.std(0)/np.sqrt(M)/base).max())

def worst_tau(tgt, k, **kw):
    p = TAU_RUN[k]
    tr, _ = nald(tgt, seed=2024, **p, **kw)
    return np.array([iact(tr[:, :, i])*p["thin"] for i in range(4)])

TOL      = 0.04
H_OP     = {"A": 0.005, "B": 0.0025}      # B needs half the step: its kinks are far denser
TAU_RUN  = {"A": dict(h=0.005,  n_steps=500_000, burn=100_000, thin=10, n_chains=48),
            "B": dict(h=0.0025, n_steps=700_000, burn=140_000, thin=20, n_chains=48)}
ALPHAS   = [2.0, 4.0, 6.0, 8.0]

In [ ]:
def variants(k):
    u = DESIGN[k]["u"]
    v = [("J0", 0.0, dict(J="none"))]
    v += [("Jdem", 4.0, dict(J="const", alpha=4.0, Ja=hat(U_DEM)))]            # reference
    v += [("Ja", a, dict(J="const", alpha=a, Ja=hat(u)))       for a in ALPHAS]
    v += [("Js", a, dict(J="state", alpha=a, s=S_STATE[k]))    for a in ALPHAS]
    return v

SEL, TAU, BIAS = {}, {}, {}
for k, tg in TARGETS.items():
    print(f"=== target {k}: {TNAME[k]}   (h = {H_OP[k]}) ===")
    print(f"  {'config':<22}{'tau_x1':>9}{'tau_x2':>9}{'tau_x3':>9}{'tau_U':>9}{'worst':>10}"
          f"{'speed':>8}{'bias':>9}")
    for tag, a, kw in variants(k):
        t   = worst_tau(tg, k, **kw)
        b, _ = stat_drift(tg, h=H_OP[k], **kw)
        TAU[(k, tag, a)], BIAS[(k, tag, a)] = t, b
        if tag == "J0": base = t.max()
        lab = {"J0": "J = 0", "Jdem": "J_a democratic a=4"}.get(tag, f"{tag} optimal a={a:g}" if tag=="Ja" else f"J_s a={a:g}")
        print(f"  {lab:<22}" + "".join(f"{x:>9.0f}" for x in t)
              + f"{t.max():>10.0f}{base/t.max():>7.2f}x{b:>9.4f}" + ("  rejected" if b > TOL else ""))
    for tag in ["Ja", "Js"]:
        ok = [a for a in ALPHAS if BIAS[(k, tag, a)] <= TOL]
        SEL[(k, tag)] = min(ok, key=lambda a: TAU[(k, tag, a)].max()) if ok else min(ALPHAS)
    print(f"  selected:  J_a alpha = {SEL[(k,'Ja')]:g}   J_s alpha = {SEL[(k,'Js')]:g}\n")

---
# 3. Convergence of the law

All chains start from the same over-dispersed point mass, $x_0=2\,\mathrm{sd}_\pi$ component-wise — a
generic off-axis point, atypical in every coordinate, so all three modes must relax. Every variant uses
the same $M$, the same $x_0$, the same step size and one $\nabla U_0$ evaluation per step, so the
horizontal axis is cost as well as time.

The ad-hoc democratic axis is carried along as a fourth curve, so the effect of the design in §1 is
visible rather than asserted.

In [ ]:
def unit_dirs(L, d, rng):
    v = rng.standard_normal((L, d))
    return v/np.linalg.norm(v, axis=1, keepdims=True)


class Discrepancy:
    # Sliced W1 and binned TV between an ensemble and a fixed exact reference sample.
    def __init__(self, Xref, sd, L=64, B=50, seed=0):
        self.sd, self.B, self.M = sd, B, Xref.shape[0]
        self.dirs = unit_dirs(L, Xref.shape[1], np.random.default_rng(seed))
        self.Pref = np.sort(Xref/sd @ self.dirs.T, axis=0)
        self.Cref = np.sort(Xref, axis=0)
        q = np.linspace(0, 1, B + 1)[1:-1]
        self.Eproj = np.quantile(self.Pref, q, axis=0)
        self.Ecoor = np.quantile(self.Cref, q, axis=0)

    def _tv(self, sorted_col, edges):
        idx = np.searchsorted(sorted_col, edges)
        cnt = np.diff(np.concatenate(([0], idx, [len(sorted_col)])))
        return 0.5*np.abs(cnt/len(sorted_col) - 1.0/self.B).sum()

    def __call__(self, X):
        P  = np.sort(X/self.sd @ self.dirs.T, axis=0)
        Cc = np.sort(X, axis=0)
        return dict(
            sw1 = float(np.abs(P - self.Pref).mean()),
            stv = float(np.mean([self._tv(P[:, l], self.Eproj[:, l]) for l in range(P.shape[1])])),
            w1  = np.abs(Cc - self.Cref).mean(0),
            tv  = np.array([self._tv(Cc[:, j], self.Ecoor[:, j]) for j in range(Cc.shape[1])]),
        )


def convergence_run(tgt, kw, h, M, disc, x0, rec_steps, seed=7):
    # M independent chains from the point mass at x0; discrepancies evaluated at rec_steps.
    X = np.tile(np.asarray(x0, float), (M, 1))
    out, done, t0 = {kk: [] for kk in ("sw1", "stv", "w1", "tv")}, 0, time.time()
    for target_step in rec_steps:
        n = target_step - done
        if n > 0:
            _, X = nald(tgt, h=h, n_steps=n, n_chains=M, seed=seed + done, x0=X,
                        store=False, block=25, **kw)
            done = target_step
        d = disc(X)
        for kk in out: out[kk].append(d[kk])
    for kk in out: out[kk] = np.asarray(out[kk])
    out["t"] = np.asarray(rec_steps)*h
    out["wall"] = time.time() - t0
    return out

In [ ]:
M, L_DIR, N_BINS = 40_000, 64, 50
CFG = {"A": dict(h=0.005,  n_steps=60_000),   # T = 300
       "B": dict(h=0.0025, n_steps=40_000)}   # T = 100

rng = np.random.default_rng(0)
REF, DISC, FLOOR, X0 = {}, {}, {}, {}
for k, tg in TARGETS.items():
    REF[k]  = tg.sample(M, rng)
    DISC[k] = Discrepancy(REF[k], SD[k], L=L_DIR, B=N_BINS, seed=11)
    X0[k]   = 2*SD[k]
    fl = [DISC[k](tg.sample(M, rng)) for _ in range(6)]
    FLOOR[k] = {kk: np.mean([f[kk] for f in fl], axis=0) for kk in ("sw1", "stv", "w1", "tv")}
    print(f"target {k}:  x0 = {np.round(X0[k],3)},  T = {CFG[k]['n_steps']*CFG[k]['h']:.0f},  h = {CFG[k]['h']}")
    print(f"            finite-sample floor   sliced W1 = {FLOOR[k]['sw1']:.5f}   sliced TV = {FLOOR[k]['stv']:.5f}")

In [ ]:
def rec_grid(n_steps, n=64):
    g = np.unique(np.round(np.logspace(0, np.log10(n_steps), n)).astype(int))
    return np.concatenate(([0], g))

def final_variants(k):
    u = DESIGN[k]["u"]
    aa, as_ = SEL[(k, "Ja")], SEL[(k, "Js")]
    return [("J0",   r"$J=0$",                                        dict(J="none")),
            ("Jdem", r"$J_a$ ad-hoc axis, $\alpha=4$",                 dict(J="const", alpha=4.0, Ja=hat(U_DEM))),
            ("Ja",   rf"$J_a$ designed axis, $\alpha={aa:g}$",         dict(J="const", alpha=aa,  Ja=hat(u))),
            ("Js",   rf"$J_s(x)$, $\alpha={as_:g}$",                   dict(J="state", alpha=as_, s=S_STATE[k]))]

LABEL, RES = {}, {}
for k, tg in TARGETS.items():
    RES[k] = {}; rec = rec_grid(CFG[k]["n_steps"])
    print(f"--- target {k}: {M:,} chains x {CFG[k]['n_steps']:,} steps ---")
    for tag, lab, kw in final_variants(k):
        LABEL[(k, tag)] = lab
        r = convergence_run(tg, kw, CFG[k]["h"], M, DISC[k], X0[k], rec)
        RES[k][tag] = r
        print(f"  {tag:<5} final sliced W1 = {r['sw1'][-1]:.5f}  sliced TV = {r['stv'][-1]:.5f}  [{r['wall']:6.1f}s]")

In [ ]:
floor_proxy = Line2D([], [], color="0.55", lw=.8, ls="--", label="finite-sample floor")
ORDER = ["J0", "Jdem", "Ja", "Js"]

fig, axes = plt.subplots(2, 2, figsize=(11.5, 7.2))
for j, k in enumerate(TARGETS):
    for i, (key, name) in enumerate([("sw1", r"sliced $\mathcal{W}_1$"), ("stv", "sliced TV")]):
        ax = axes[i, j]; fl = FLOOR[k][key]
        ax.axhspan(0, fl, color="0.88", zorder=0)
        ax.axhline(fl, color="0.55", lw=.8, ls="--", zorder=1)
        for tag in ORDER:
            r = RES[k][tag]
            ax.loglog(np.maximum(r["t"], r["t"][1]/2), r[key], color=COL[tag],
                      lw=1.6 if tag in ("Ja", "Js") else 1.2,
                      ls="--" if tag == "Jdem" else "-", label=LABEL[(k, tag)])
        ax.set_xlabel("simulated time $t$   (= cost: one $\\nabla U_0$/step, same $h$ throughout)")
        ax.set_ylabel(f"{name}$(\\,\\mathrm{{Law}}(X_t),\\ \\pi\\,)$")
        ax.set_title(f"Target {k} — {TNAME[k]}   ($M={M:,}$ chains)")
        h_, l_ = ax.get_legend_handles_labels()
        ax.legend(h_ + [floor_proxy], l_ + [floor_proxy.get_label()], fontsize=7.5, loc="lower left")
fig.suptitle("Convergence of the law of $X_t$ to $\\pi \\propto e^{-U}$ from a common point mass", y=1.005)
fig.tight_layout(); plt.show()

In [ ]:
def time_to(r, key, level):
    y, t = r[key], r["t"]
    for i in range(1, len(y)):
        if y[i] <= level:
            if y[i-1] <= level: return float(t[i-1])
            if t[i-1] <= 0 or y[i] <= 0 or y[i-1] <= 0: return float(t[i])
            lo, hi = np.log(y[i-1]), np.log(y[i])
            if hi >= lo: return float(t[i])
            f = (np.log(level) - lo)/(hi - lo)
            return float(np.exp(np.log(t[i-1])*(1-f) + np.log(t[i])*f))
    return np.nan

print("time for the law to reach a multiple of the finite-sample floor (speed-up vs J = 0)\n")
TT = {}
for k in TARGETS:
    for key, name in [("sw1", "sliced W1"), ("stv", "sliced TV")]:
        print(f"  target {k}  {name}")
        for mult in [8, 4, 2]:
            lvl = mult*FLOOR[k][key]
            row = {tag: time_to(RES[k][tag], key, lvl) for tag in ORDER}
            TT[(k, key, mult)] = row; base = row["J0"]
            s = "   ".join(f"{tag}: {row[tag]:7.2f}" + (f" ({base/row[tag]:5.2f}x)" if tag != "J0" else "         ")
                           for tag in ORDER)
            print(f"    {mult}x floor:  {s}")
        print()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.6))
mults = [8, 4, 2]; w = 0.2
for j, key in enumerate(["sw1", "stv"]):
    ax = axes[j]; labels = []
    for i, (k, mult) in enumerate([(k, m) for k in TARGETS for m in mults]):
        labels.append(f"{k}\n{mult}x")
        base = TT[(k, key, mult)]["J0"]
        for o, tag in enumerate(ORDER):
            ax.bar(i + (o-1.5)*w, base/TT[(k, key, mult)][tag], w, color=COL[tag],
                   label=LABEL[(k, tag)].split(",")[0] if i == 0 else None)
    ax.axhline(1.0, color="0.5", lw=.8, ls=":")
    ax.set_xticks(np.arange(len(labels))); ax.set_xticklabels(labels, fontsize=7.5)
    ax.set_ylabel("speed-up in time-to-accuracy")
    ax.set_title({"sw1": r"sliced $\mathcal{W}_1$", "stv": "sliced TV"}[key])
    if j == 0: ax.legend(fontsize=7.5)
fig.suptitle("Time for the law of $X_t$ to reach a fixed multiple of the finite-sample floor", y=1.04)
fig.tight_layout(); plt.show()

---
# 4. Per-coordinate breakdown

$x_1$ is the slow mode in both targets, $x_3$ the fast one. This is where the difference between the
two perturbations shows up: a designed constant $J_a$ equalises the modes and therefore attacks $x_1$
directly, whereas $J_s(x)\nabla U_0=s\,(x\times\nabla U_0)$ **vanishes identically wherever
$x\parallel\nabla U_0$** — the principal axes for target A — so it stirs the fast directions and leaves
the slow one largely alone.

In [ ]:
for k in TARGETS:
    fig, axes = plt.subplots(2, 3, figsize=(12, 6))
    for j in range(3):
        for i, (key, name) in enumerate([("w1", r"$\mathcal{W}_1$"), ("tv", "TV")]):
            ax = axes[i, j]; fl = FLOOR[k][key][j]
            ax.axhspan(0, fl, color="0.88", zorder=0)
            ax.axhline(fl, color="0.55", lw=.8, ls="--", zorder=1)
            for tag in ORDER:
                r = RES[k][tag]
                ax.loglog(np.maximum(r["t"], r["t"][1]/2), r[key][:, j], color=COL[tag],
                          lw=1.5 if tag in ("Ja", "Js") else 1.1,
                          ls="--" if tag == "Jdem" else "-", label=LABEL[(k, tag)])
            ax.set_title(f"$x_{j+1}$  ({name}),  sd$_\\pi$ = {SD[k][j]:.2f}", fontsize=9)
            ax.set_xlabel("simulated time $t$"); ax.set_ylabel(f"{name} to $\\pi$")
            if i == 0 and j == 0: ax.legend(fontsize=7, loc="lower left")
    fig.suptitle(f"Target {k} — {TNAME[k]}: per-coordinate convergence", y=1.01)
    fig.tight_layout(); plt.show()

---
## What changed, and what did not

**The axis was the binding constraint on $J_a$.** An arbitrary skew perturbation gets a small fraction
of what is available: the democratic axis $u=(1,1,1)/\sqrt3$ saturates at about $2.6\times$ the
reversible spectral gap on target A, while the ceiling $\mathrm{tr}(S)/3$ is $15.3\times$. Choosing
$u$ to equalise the relaxation rates captures most of that, at **no cost in accuracy** — the measured
discretisation bias at the designed axis is no worse than at the arbitrary one, because $\alpha$ did
not have to grow.

**$J_s$ is limited by its own geometry, not by tuning.** Its perturbation
$\alpha c=(\alpha s\,a\nabla U_0)\times x$ vanishes wherever $x\parallel\nabla U_0$, which for an
elliptical target is exactly the principal axes — the slow manifold. No choice of $\alpha$ or $s$
removes that degeneracy, since only the product $\alpha s$ matters and the null set is independent of
it. So $J_s$ accelerates the fast coordinates strongly and the slow one weakly, and since overall
convergence is governed by the slowest mode, its gain in $\mathcal{W}_1$ and TV is the more modest of
the two. That is a property of the prescribed $J_s$, not an artefact of the experiment.

**Things that were tried and did not help**, recorded so they are not re-attempted:

* *Leimkuhler–Matthews integration* (the averaged-noise scheme, which is superconvergent for
  overdamped Langevin) to buy accuracy and hence allow larger $\alpha$. It gave no improvement beyond
  the diagnostic's noise floor here: its superconvergence requires **additive** noise, whereas NALD has
  the state-dependent diffusion $\sqrt{2a(x)}$, and in any case the dominant error at large $\alpha$
  comes from the drift/rotation splitting rather than from the noise discretisation.
* *Pushing $\alpha$ higher.* The worst-mode $\tau$ saturates — on target A it improves from
  $7.6\times$ at $\alpha=4$ to only $9.9\times$ at $\alpha=12$ — which is exactly what the
  $\mathrm{tr}(S)/3$ ceiling predicts, while the bias keeps growing. Past the ceiling, extra stirring
  redistributes rates between modes instead of raising the slowest.